In [1]:
try:
    import torch
    import torchvision
    import torchinfo
    assert int(torch.__version__.split(".")[1]) >= 13
    assert int(torchvision.__version__.split(".")[1]) >= 13
except:
    print(f"[info] installing")
    !pip3 install torch torchvision torchinfo
    import torch
    import torchvision
    import torchinfo
    print(f"pytorch : {torch.__version__}")
    print(f"pytorch : {torchvision.__version__}")



c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\cuda\__init__.py:64: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
import matplotlib.pyplot as plt
from torchinfo import summary
from torchvision import transforms
from going_modular import data_setup,engine
from helper_functions import download_data,set_seeds,plot_loss_curves

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [4]:
data_20_percent_path = download_data("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip","pizza_steak_sushi_20_percent")
data_20_percent_path

[INFO] data\pizza_steak_sushi_20_percent directory exists, skipping download.


WindowsPath('data/pizza_steak_sushi_20_percent')

In [5]:
train_dir = data_20_percent_path / "train"
test_dir = data_20_percent_path / "test"

In [6]:
# creating effnet b2 
weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
effnet_transform = weights.transforms()
effnetb2 = torchvision.models.efficientnet_b2(weights=weights)
for param in effnetb2.parameters():
    param.requires_grad = False

In [7]:
effnetb2.classifier
from torch import nn

In [8]:
effnetb2.classifier = nn.Sequential(
    nn.Dropout(p=0.3,inplace=True),
    nn.Linear(in_features = 1408, out_features = 3)
)

In [9]:
def create_effnetb2_model(num_classes:int=3,
                          seed:int=42):

    weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
    transforms = weights.transforms()
    model = torchvision.models.efficientnet_b2(weights=weights)

    for param in model.parameters():
        param.requires_grad = False


    torch.manual_seed(seed)

    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3,inplace=True),
        nn.Linear(in_features = 1408, out_features=num_classes)
    )

    return model, transforms

In [10]:
effnetb2,effnet_transforms = create_effnetb2_model(3,42)

In [11]:
summary(effnetb2, 
        input_size=(1, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [1, 3, 224, 224]     [1, 3]               --                   Partial
├─Sequential (features)                                      [1, 3, 224, 224]     [1, 1408, 7, 7]      --                   False
│    └─Conv2dNormActivation (0)                              [1, 3, 224, 224]     [1, 32, 112, 112]    --                   False
│    │    └─Conv2d (0)                                       [1, 3, 224, 224]     [1, 32, 112, 112]    (864)                False
│    │    └─BatchNorm2d (1)                                  [1, 32, 112, 112]    [1, 32, 112, 112]    (64)                 False
│    │    └─SiLU (2)                                         [1, 32, 112, 112]    [1, 32, 112, 112]    --                   --
│    └─Sequential (1)                                        [1, 32, 112, 112]    [1, 1

In [12]:
from going_modular import data_setup ,engine

train_dataloader_effnetb2,test_dataloader_effnetb2,class_name=data_setup.create_dataloaders(train_dir=train_dir,
                                                                                            test_dir=test_dir,
                                                                                            transform=effnet_transform,
                                                                                            batch_size=32)

In [15]:
optimizer = torch.optim.Adam(params=effnetb2.parameters(),lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()
set_seeds()
effnetb2_results = engine.train(effnetb2,
                                train_dataloader_effnetb2,
                                test_dataloader_effnetb2,
                                optimizer,
                                loss_fn,
                                10,
                                device)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:04<?, ?it/s]


KeyboardInterrupt: 

In [16]:
from going_modular.utils import save_model
save_model(effnetb2,"models","naww.pth")

[INFO] Saving model to: models\naww.pth


In [17]:
from pathlib import Path
model_path = Path("models")
model_name = "naww.pth"
effnetb2.load_state_dict(torch.load(model_path / model_name,))


<All keys matched successfully>

In [18]:
from helper_functions import plot_loss_curves
plot_loss_curves(effnetb2_results)

NameError: name 'effnetb2_results' is not defined

In [19]:
model = (model_path / model_name)
print(f"size : {model.stat().st_size // (1024*1024)} MB")

size : 29 MB


In [20]:
effnetb2_total_params = sum(torch.numel(param) for param in effnetb2.parameters())
effnetb2_total_params

7705221

In [ ]:
effnetb2_stat = {}